# FraudLens Medallion Exploration

Explore the **Silver** and **Gold** layers for FraudLens using PySpark.

- Inspect schemas and row counts
- Compute basic descriptive stats
- Look at example suspicious patterns (velocity, spikes, impossible travel)


In [ ]:
from pathlib import Path
import sys

from pyspark.sql import SparkSession

# Robustly find the project root by walking up until we see a src/ directory
project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root.parent != project_root:
    project_root = project_root.parent

src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from fraud_lens.ingest import load_paths_config

spark = (
    SparkSession.builder.appName("FraudLens-Explore-Silver-Gold")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

# Use project_root/data as the base for all layers
config = load_paths_config()
data_cfg = config.get("data", {})

silver_rel = data_cfg.get("silver", "data/silver")
gold_rel = data_cfg.get("gold", "data/gold")

silver_path = (project_root / silver_rel).resolve()
gold_path = (project_root / gold_rel).resolve()


In [ ]:
# Load Silver and Gold DataFrames

from pyspark.sql import functions as F

silver_df = spark.read.parquet(str(silver_path))
print("Silver:")
print(f"  Rows: {silver_df.count():,}")
silver_df.printSchema()

# Gold might not exist yet if pipeline hasn't been run
try:
    gold_df = spark.read.parquet(str(gold_path))
    print("\nGold:")
    print(f"  Rows: {gold_df.count():,}")
    gold_df.printSchema()
except Exception as e:
    gold_df = None
    print("\nGold layer not found yet — run scripts/run_pipeline.py first.")
    print(f"Error: {e}")


In [ ]:
# Basic Silver exploration: sample, stats, anomaly counts

print("Sample Silver rows:")
display(silver_df.limit(10))

print("\nBasic numeric stats (amount, latitude, longitude):")
numeric_cols = ["amount", "latitude", "longitude"]
for col in numeric_cols:
    silver_df.select(col).describe().show()

print("\nAnomaly type distribution (from generator):")
(
    silver_df.groupBy("anomaly_type")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)


In [ ]:
# Gold feature exploration: velocity, amount spikes, impossible travel

if gold_df is None:
    print("Gold DataFrame is not loaded; skip this section.")
else:
    print("Gold feature columns:")
    feature_cols = [
        "time_since_last_tx_minutes",
        "tx_count_last_1h",
        "tx_count_last_24h",
        "amount_zscore",
        "is_amount_spike",
        "distance_from_prev_km",
        "hours_since_prev",
        "speed_from_prev_kmh",
    ]
    print(feature_cols)

    print("\nSample Gold rows (features + label):")
    display(
        gold_df.select(
            "transaction_id",
            "card_id",
            "event_time",
            "amount",
            "anomaly_type",
            *feature_cols,
        ).limit(20)
    )
    print(f"\n{gold_df.count()}")
    num_cards = gold_df.select("card_id").distinct().count()
    print(f"Distinct card_ids: {num_cards}")
    
    print("\nTop potential spending spikes (highest amount_zscore):")
    (
        gold_df.orderBy(F.desc("amount_zscore"))
        .select("card_id", "event_time", "amount", "amount_zscore", "is_amount_spike", "anomaly_type")
        .limit(20)
        .show(truncate=False)
    )

    print("\nTop potential impossible travel cases (highest speed_from_prev_kmh):")
    (
        gold_df.orderBy(F.desc("speed_from_prev_kmh"))
        .select(
            "card_id",
            "event_time",
            "distance_from_prev_km",
            "hours_since_prev",
            "speed_from_prev_kmh",
            "anomaly_type",
        )
        .limit(20)
        .show(truncate=False)
    )


In [ ]:
# Optional: per-card time series preview for a single card

example_card = (
    silver_df.select("card_id")
    .groupBy("card_id")
    .count()
    .orderBy(F.desc("count"))
    .limit(1)
    .collect()[0]["card_id"]
)

print(f"Example card_id: {example_card}")

silver_card = (
    silver_df.filter(F.col("card_id") == example_card)
    .orderBy("event_time")
)


print("Silver time series for example card:")
display(silver_card.limit(50))

if gold_df is not None:
    gold_card = (
        gold_df.filter(F.col("card_id") == example_card)
        .orderBy("event_time")
    )
    print("\nGold features time series for example card:")
    display(
        gold_card.select(
            "event_time",
            "amount",
            "tx_count_last_1h",
            "tx_count_last_24h",
            "amount_zscore",
            "is_amount_spike",
            "distance_from_prev_km",
            "speed_from_prev_kmh",
            "anomaly_type",
        ).limit(100)
    )


## Sanity checks: rule-based vs. synthetic labels

The rules below mirror `docs/data_sanity_rules.md` and check how well the synthetic labels (`anomaly_type`) align with feature-based definitions for spending spikes and impossible travel.


In [ ]:
# Rule-based sanity checks on Gold

from pyspark.sql import functions as F

if gold_df is None:
    print("Gold DataFrame is not loaded; skip sanity checks.")
else:
    # Rule A: spending spikes vs amount_zscore
    spike_threshold = 3.0

    spike_stats = (
        gold_df
        .select(
            "anomaly_type",
            (F.col("amount_zscore") >= spike_threshold).alias("is_spike_by_rule"),
        )
        .groupBy("anomaly_type")
        .agg(
            F.count("*").alias("rows"),
            F.sum(F.col("is_spike_by_rule").cast("int")).alias("spike_by_rule"),
        )
        .withColumn(
            "spike_by_rule_frac",
            F.col("spike_by_rule") / F.col("rows"),
        )
        .orderBy("anomaly_type")
    )

    print("Rule A – spending spikes vs amount_zscore (threshold =", spike_threshold, "):")
    spike_stats.show(truncate=False)

    # Rule B: impossible travel vs speed_from_prev_kmh
    speed_threshold = 1000.0

    travel_stats = (
        gold_df
        .select(
            "anomaly_type",
            (F.col("speed_from_prev_kmh") > speed_threshold).alias("is_impossible_by_rule"),
        )
        .groupBy("anomaly_type")
        .agg(
            F.count("*").alias("rows"),
            F.sum(F.col("is_impossible_by_rule").cast("int")).alias("impossible_by_rule"),
        )
        .withColumn(
            "impossible_by_rule_frac",
            F.col("impossible_by_rule") / F.col("rows"),
        )
        .orderBy("anomaly_type")
    )

    print("\nRule B – impossible travel vs speed_from_prev_kmh (threshold =", speed_threshold, "):")
    travel_stats.show(truncate=False)


In [ ]:
# ── Visualizations: Gold-layer feature distributions & anomaly patterns ──

import matplotlib.pyplot as plt
from pyspark.sql import functions as F

if gold_df is None:
    print("Gold DataFrame not loaded — skipping all plots.")
else:
    # ── constants ──
    ANOMALY_FRACTIONS = {
        "none": 0.05,
        "spending_spike": 1.0,
        "impossible_travel": 1.0,
    }
    ANOMALY_COLORS = {
        "none": "#1f77b4",
        "spending_spike": "#ff7f0e",
        "impossible_travel": "#d62728",
    }
    ANOMALY_ORDER = ["none", "spending_spike", "impossible_travel"]

    # ── helper: overlay histogram split by anomaly_type ──
    def _overlay_hist(ax, pdf, col, xlabel, title, bins=50):
        for atype in ANOMALY_ORDER:
            subset = pdf[pdf["anomaly_type"] == atype]
            if not subset.empty:
                ax.hist(
                    subset[col],
                    bins=bins,
                    alpha=0.55,
                    label=atype,
                    color=ANOMALY_COLORS[atype],
                )
        ax.set_yscale("log")
        ax.set_xlabel(xlabel)
        ax.set_ylabel("count (log)")
        ax.set_title(title)
        ax.legend()

    # ── prepare data (one Spark pass each) ──
    amount_sample = (
        gold_df.select("amount", "amount_zscore", "anomaly_type")
        .sampleBy("anomaly_type", fractions=ANOMALY_FRACTIONS, seed=42)
    )
    pct = amount_sample.selectExpr(
        "percentile_approx(amount, array(0.01, 0.99), 10000)       as amt_q",
        "percentile_approx(amount_zscore, array(0.01, 0.99), 10000) as z_q",
    ).first()
    amt_q, z_q = pct["amt_q"], pct["z_q"]

    overlay_pdf = (
        amount_sample
        .withColumn(
            "amount_clip",
            F.greatest(F.lit(amt_q[0]), F.least(F.col("amount"), F.lit(amt_q[1]))),
        )
        .withColumn(
            "zscore_clip",
            F.greatest(F.lit(z_q[0]), F.least(F.col("amount_zscore"), F.lit(z_q[1]))),
        )
        .toPandas()
    )

    speed_full = gold_df.select(
        "speed_from_prev_kmh", "anomaly_type",
    ).where(F.col("speed_from_prev_kmh").isNotNull())

    spd_q = speed_full.selectExpr(
        "percentile_approx(speed_from_prev_kmh, array(0.01, 0.99), 10000) as q"
    ).first()["q"]

    imp_spd_q = (
        gold_df.select("speed_from_prev_kmh")
        .where(
            (F.col("speed_from_prev_kmh").isNotNull())
            & (F.col("anomaly_type") == "impossible_travel")
        )
        .selectExpr(
            "percentile_approx(speed_from_prev_kmh, array(0.01, 0.995), 10000) as q"
        )
        .first()["q"]
    )

    speed_pdf = (
        speed_full
        .sampleBy("anomaly_type", fractions=ANOMALY_FRACTIONS, seed=43)
        .withColumn(
            "speed_clip",
            F.greatest(
                F.lit(spd_q[0]),
                F.least(F.col("speed_from_prev_kmh"), F.lit(spd_q[1])),
            ),
        )
        .withColumn(
            "imp_speed_display",
            F.when(
                F.col("anomaly_type") == "impossible_travel",
                F.greatest(
                    F.lit(imp_spd_q[0]),
                    F.least(F.col("speed_from_prev_kmh"), F.lit(imp_spd_q[1])),
                ),
            ).otherwise(F.col("speed_clip")),
        )
        .toPandas()
    )

    imp_pdf = (
        gold_df.select("speed_from_prev_kmh")
        .where(
            (F.col("speed_from_prev_kmh").isNotNull())
            & (F.col("anomaly_type") == "impossible_travel")
        )
        .withColumn(
            "speed_display",
            F.greatest(
                F.lit(imp_spd_q[0]),
                F.least(F.col("speed_from_prev_kmh"), F.lit(imp_spd_q[1])),
            ),
        )
        .toPandas()
    )

    dist_pdf = (
        gold_df.select("distance_from_prev_km", "hours_since_prev", "anomaly_type")
        .where(F.col("hours_since_prev") > 0)
        .sample(withReplacement=False, fraction=0.1, seed=44)
        .toPandas()
    )

    # ── build 3×2 subplot grid ──
    fig, axes = plt.subplots(3, 2, figsize=(16, 18))
    fig.suptitle(
        "FraudLens — Gold Layer Feature Distributions",
        fontsize=16, fontweight="bold", y=0.98,
    )

    # (0,0) amount distribution
    _overlay_hist(
        axes[0, 0], overlay_pdf, "amount_clip",
        f"amount (clipped 1st–99th pct: [{amt_q[0]:.1f}, {amt_q[1]:.1f}])",
        "Amount distribution by anomaly_type",
    )

    # (0,1) z-score distribution
    _overlay_hist(
        axes[0, 1], overlay_pdf, "zscore_clip",
        f"amount_zscore (clipped [{z_q[0]:.2f}, {z_q[1]:.2f}])",
        "Amount z-score distribution by anomaly_type",
    )

    # (1,0) speed distribution with display clipping for readability
    ax_speed = axes[1, 0]
    for atype in ANOMALY_ORDER:
        subset = speed_pdf[speed_pdf["anomaly_type"] == atype]
        if subset.empty:
            continue
        ax_speed.hist(
            subset["imp_speed_display"],
            bins=50,
            alpha=0.55,
            label=atype,
            color=ANOMALY_COLORS[atype],
        )
    ax_speed.set_yscale("log")
    ax_speed.set_xlabel("speed_from_prev_kmh (display-clipped for readability)")
    ax_speed.set_ylabel("count (log)")
    ax_speed.set_title(
        f"Travel speed distribution by anomaly_type | impossible_travel clipped at 99.5th pct ({imp_spd_q[1]:.1f})"
    )
    ax_speed.legend()

    # (1,1) impossible_travel speeds with display clipping
    ax_imp = axes[1, 1]
    if imp_pdf.empty:
        ax_imp.text(
            0.5, 0.5, "No impossible_travel data",
            transform=ax_imp.transAxes, ha="center", fontsize=12,
        )
    else:
        ax_imp.hist(
            imp_pdf["speed_display"],
            bins=40,
            alpha=0.8,
            color=ANOMALY_COLORS["impossible_travel"],
        )
        ax_imp.axvline(
            imp_spd_q[1],
            color="black",
            linestyle="--",
            linewidth=1,
            label="99.5th pct clip",
        )
        ax_imp.legend()
    ax_imp.set_xlabel("speed_from_prev_kmh (display-clipped)")
    ax_imp.set_ylabel("count")
    ax_imp.set_title("Impossible travel speed distribution")

    # (2,0) distance vs time hexbin
    ax_hex = axes[2, 0]
    hb = ax_hex.hexbin(
        dist_pdf["hours_since_prev"],
        dist_pdf["distance_from_prev_km"],
        gridsize=40, cmap="viridis", mincnt=1,
    )
    ax_hex.set_xlabel("hours_since_prev")
    ax_hex.set_ylabel("distance_from_prev_km")
    ax_hex.set_title("Distance vs time between transactions")
    fig.colorbar(hb, ax=ax_hex, label="count")

    # (2,1) unused — hide
    axes[2, 1].axis("off")

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


In [ ]:
# Debug view: inspect the most extreme impossible_travel rows

from pyspark.sql import functions as F

if gold_df is None:
    print("Gold DataFrame not loaded — skipping impossible_travel debug view.")
else:
    (
        gold_df.select(
            "transaction_id",
            "card_id",
            "event_time",
            "distance_from_prev_km",
            "hours_since_prev",
            "speed_from_prev_kmh",
        )
        .where(F.col("anomaly_type") == "impossible_travel")
        .orderBy(F.col("speed_from_prev_kmh").desc())
        .show(15, truncate=False)
    )

In [ ]:
# Cleanup (optional when done)

spark.stop()
